# 🎯 Joint Linear Regression Fitter — 3 equations, 6 unknowns, 3 data sources

Fits the **6 unknown parameters that are shared across 3 equations** using all 3 data sources at once (one joint scikit-learn fit on the stacked data).

**How to use**
1. 🟨 **Step 1 — the cell below:** type your 3 equations
2. 🟨 **Step 2:** paste your 3 data sources
3. **Run All** — the model fits, prints the 6 fitted unknowns + per-equation metrics, and draws one plot per equation

> Only the two 🟨 yellow cells are meant to be edited — everything below them is machinery.

> Needs `numpy pandas scikit-learn matplotlib`. If any is missing, run `!pip install numpy pandas scikit-learn matplotlib` in a cell.

<div style="background-color:#fef3c7; border-left:6px solid #f59e0b; padding:10px 14px; border-radius:6px;">
<strong>🟨 STEP 1 — INPUT YOUR 3 EQUATIONS</strong><br/>
Edit the next code cell: one tuple per equation — <code>(equation, target column, data source)</code>.<br/>
The parameters (<code>a</code>…<code>f</code>) are <strong>shared</strong> across the three equations and fitted jointly.
</div>

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   🟨 EDIT HERE — STEP 1: YOUR 3 EQUATIONS  🟨                   ║
# ╚══════════════════════════════════════════════════════════════════╝
# One tuple per equation:  (equation, target column, data source)
#
#   • PARAMETERS = single letters to be fitted — 6 in total, SHARED across
#     the three equations (each equation may use any of them).
#   • VARIABLES  = names that match columns in that equation's data source
#     (e.g. x, t, w).
#   • TARGET     = the column of that source you are predicting.
#   • Each equation must be LINEAR IN ITS PARAMETERS, e.g.:
#         "a + b*x" | "a + b*log(x)" | "a*sin(x) + b*cos(x)"   OK
#         "a*exp(b*x)"   ✗ b is inside exp() → use scipy.optimize.curve_fit
#     (Note: avoid using "e" or "pi" as parameter names — they are reserved.)
#   • Any equation may use any source — just change the source name in its
#     tuple (e.g. two equations can share the same source).

EXPECTED_UNKNOWNS = 6   # how many unknown parameters the equations share

EQUATIONS = [
    # (equation,                target, data source)
    ("a + b*x + c*x**2",        "y", "source1"),
    ("a + d*t + e*t**2",        "y", "source2"),
    ("c*w + f*sin(w) + e*w",    "y", "source3"),
]

<div style="background-color:#fef3c7; border-left:6px solid #f59e0b; padding:10px 14px; border-radius:6px;">
<strong>🟨 STEP 2 — INPUT YOUR 3 DATA SOURCES</strong><br/>
Edit the next code cell: one dict per source, keyed by the names used in Step 1. Sources may have different lengths.
</div>

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   🟨 EDIT HERE — STEP 2: YOUR 3 DATA SOURCES  🟨                ║
# ╚══════════════════════════════════════════════════════════════════╝
# One dict per source; the keys are the column names used in Step 1.
# (Sample data below, true params: a=1.5, b=2, c=3, d=4, e=0.5, f=6)
DATA_SOURCES = {
    "source1": {  # y = 1.5 + 2·x + 3·x² + noise
        "x": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "y": [6.6, 17.3, 34.7, 57.2, 86.9, 121.3, 162.8, 209.2, 262.9, 321.1],
    },
    "source2": {  # y = 1.5 + 4·t + 0.5·t² + noise
        "t": [1, 2, 3, 4, 5, 6, 7, 8],
        "y": [6.1, 11.4, 18.2, 25.4, 34.2, 43.7, 53.8, 65.7],
    },
    "source3": {  # y = 3.5·w + 6·sin(w) + noise
        "w": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0],
        "y": [4.7, 8.6, 11.3, 12.4, 12.4, 11.3, 10.2, 9.5, 9.8, 11.8, 15.0, 19.4],
    },
}

# ...or load real data from CSVs instead (one column per variable):
# import pandas as pd
# df1 = pd.read_csv("ticker_A.csv")
# DATA_SOURCES = {
#     "source1": {col: df1[col].to_numpy() for col in df1.columns},
#     # ...same for source2, source3
# }

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   STEP 3: THE MODEL — no need to edit anything below this line   ║
# ╚══════════════════════════════════════════════════════════════════╝
import re

import numpy as np
from sklearn.linear_model import LinearRegression

# functions you may use inside the equations
ns = {
    "sin": np.sin, "cos": np.cos, "tan": np.tan,
    "exp": np.exp, "log": np.log, "log2": np.log2, "log10": np.log10,
    "sqrt": np.sqrt, "abs": np.abs, "pow": np.power,
    "pi": np.pi, "e": np.e,
}

# ---- 1. Parse the three equations -----------------------------------------
parsed = []          # (equation text, target, source) per equation
all_ids = set()
targets = set()
for i, (eq, target, source) in enumerate(EQUATIONS, start=1):
    if source not in DATA_SOURCES:
        raise ValueError(f"Equation {i} uses '{source}' but DATA_SOURCES has no such key.")
    if target not in DATA_SOURCES[source]:
        raise ValueError(f"Equation {i}: column '{target}' missing from '{source}'.")
    eq = eq.strip()
    if "=" in eq:
        eq = eq.split("=", 1)[1].strip()
    all_ids |= set(re.findall(r"[A-Za-z_]\w*", eq))
    targets.add(target)
    parsed.append((eq, target, source))

# data-column names actually used by the equations (never treated as parameters)
var_names = {name
             for _, _, source in parsed
             for name in DATA_SOURCES[source] if name in all_ids}

# parameters = the remaining identifiers, shared across all equations
params = sorted(all_ids - set(ns) - var_names - targets)
print(f"Parameters     : {params}  ({len(params)} detected, expected {EXPECTED_UNKNOWNS})")
if len(params) != EXPECTED_UNKNOWNS:
    print(f"⚠ Expected {EXPECTED_UNKNOWNS} unknowns but found {len(params)} — check your equations.")

# ---- 2. Build the stacked design matrix (one column per parameter) --------
def _term_for(eq_text, source, param):
    local = dict(ns)
    for name in DATA_SOURCES[source]:
        local[name] = np.asarray(DATA_SOURCES[source][name], dtype=float)
    for p in params:
        local[p] = 1.0 if p == param else 0.0
    return np.asarray(eval(eq_text, {"__builtins__": {}}, local), dtype=float)

X_blocks, y_blocks = [], []
for eq_text, target, source in parsed:
    X_blocks.append(np.column_stack([_term_for(eq_text, source, p) for p in params]))
    y_blocks.append(np.asarray(DATA_SOURCES[source][target], dtype=float))

X = np.vstack(X_blocks)
y = np.concatenate(y_blocks)
print(f"Total samples  : {len(y)}")

# ---- 3. One joint fit across all equations and sources ---------------------
model = LinearRegression(fit_intercept=False)  # the 6 unknowns are the columns of X
model.fit(X, y)

# ---- 4. Fitted unknowns + per-equation metrics -----------------------------
def _r2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

def _rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print("\nFitted unknowns:")
for p, c in zip(params, model.coef_):
    print(f"  {p} = {c:.4g}")

print("\nPer-equation fit (using the shared parameters):")
eq_summaries = []
for k, ((eq_text, target, source), Xb, yb) in enumerate(zip(parsed, X_blocks, y_blocks), start=1):
    pred = model.predict(Xb)
    eq_summaries.append((k, source, target, yb, pred))
    print(f"  Eq {k} [{source}] : R² = {_r2(yb, pred):.4f}   RMSE = {_rmse(yb, pred):.4f}")

print(f"\nOverall R² = {_r2(y, model.predict(X)):.4f}")

# ---- 5. Sanity check: are the equations really linear in the parameters? ---
ok = True
for k, (eq_text, target, source) in enumerate(parsed, start=1):
    local = dict(ns)
    for name in DATA_SOURCES[source]:
        local[name] = np.asarray(DATA_SOURCES[source][name], dtype=float)
    for p, c in zip(params, model.coef_):
        local[p] = c
    recon = np.asarray(eval(eq_text, {"__builtins__": {}}, local), dtype=float)
    Xb = np.column_stack([_term_for(eq_text, source, p) for p in params])
    if not np.allclose(recon, Xb @ model.coef_, atol=1e-6):
        ok = False
        print(f"⚠ Equation {k} ('{eq_text}') looks NON-linear in its parameters.")
if ok:
    print("\n✓ Linear-in-parameters check passed for all equations — the fit is valid.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   STEP 4: DIAGNOSTIC PLOTS — one per equation                    ║
# ╚══════════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e",
    "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#898781",
    "ytick.color": "#898781",
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(1, len(eq_summaries), figsize=(5 * len(eq_summaries), 4.2))

for ax, (k, source, target, yb, pred) in zip(axes, eq_summaries):
    ax.scatter(yb, pred, s=24, alpha=0.8, color="#2a78d6",
               edgecolors="white", linewidths=0.5)
    lo, hi = min(yb.min(), pred.min()), max(yb.max(), pred.max())
    pad = 0.05 * (hi - lo)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
            ls="--", lw=1.2, color="#c3c2b7")
    ax.set_xlabel(f"Actual {target}")
    ax.set_ylabel(f"Predicted {target}")
    ax.set_title(f"Equation {k} — {source}")

fig.tight_layout()
plt.show()

In [1]:
import torch
print(torch.__version__)

2.13.0+cpu


# 🧠 PART II — solving ODEs with PyTorch (scalar + system + fitting)

Every step of the integrators below is plain torch math, so the whole simulated trajectory is one differentiable computation graph. Two payoffs:

- solve any `dz/dt = f(t, z)` with autograd tracking (Euler or RK4);
- backprop *through* the solver → learn unknown ODE parameters straight from data (cells 3–4 do this — the same job as the regression fit in PART I, but the ODE is **not** linear in its parameters, so it needs gradient descent, not `LinearRegression`).

The cells:

1. `solve_ode` machinery + the scalar case `dz/dt = p·z + a·u`, checked against its analytic solution `z(t) = (z₀ + a·u/p)·e^(p·t) − a·u/p`.
2. A **generalised system** of ODEs — `dz/dt = A·z + B·u(t)`, an N-dim coupled state with an M-dim time-varying input. Written in matrix form, so any N and M work — checked against an RK4 reference on a 10× finer grid.
3. Fitter tie-in: recover the true `p`, `a` from noisy measurements with Adam.
4. A **generalised mixed** system — N coupled ODE state equations plus N linear algebraic output equations (`yᵢ = cᵢ·zᵢ + d`) — a hidden state-space model. All of `A, B, c, d` are recovered from noisy outputs while the hidden state is never observed.

> Needs `torch matplotlib` (same environment as PART I's plot cell).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   PART II · 1 — PyTorch ODE solver, scalar:  dz/dt = p·z + a·u  ║
# ╚══════════════════════════════════════════════════════════════════╝
# The machinery (solve_ode) below is used by all three cells of this part.
import matplotlib.pyplot as plt
import torch

plt.rcParams.update({   # same style as the regression plots in PART I
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "axes.titlecolor": "#0b0b0b", "xtick.color": "#898781",
    "ytick.color": "#898781", "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8, "axes.grid": True,
    "axes.spines.top": False, "axes.spines.right": False,
})

# ---- the machinery: a differentiable Euler / RK4 integrator -----------------
def solve_ode(f, z0, t, *params, method="rk4"):
    """Integrate dz/dt = f(t, z, *params) on a uniform grid t.

    Every step is plain torch math, so autograd flows THROUGH the solver —
    that is what lets the last cell learn p and a straight from data.
    """
    dt = t[1] - t[0]
    z, zs = z0, [z0]
    for ti in t[:-1]:
        if method == "euler":
            z = z + dt * f(ti, z, *params)
        elif method == "rk4":
            k1 = f(ti,          z,              *params)
            k2 = f(ti + dt / 2, z + dt / 2 * k1, *params)
            k3 = f(ti + dt / 2, z + dt / 2 * k2, *params)
            k4 = f(ti + dt,     z + dt * k3,     *params)
            z = z + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        else:
            raise ValueError(f"unknown method {method!r}")
        zs.append(z)
    return torch.stack(zs)

# ---- the ODE and its analytic solution ---------------------------------------
def rhs_scalar(t, z, p, a, u):      # dz/dt = p·z + a·u   (constant input u)
    return p * z + a * u

p, a, u = -0.5, 1.0, 2.0            # p < 0 → stable; steady state z* = −a·u/p = 4
z0 = torch.tensor(1.0)

t = torch.linspace(0, 10, 101)      # coarse grid, dt = 0.1

z_euler = solve_ode(rhs_scalar, z0, t, p, a, u, method="euler")
z_rk4   = solve_ode(rhs_scalar, z0, t, p, a, u, method="rk4")
z_exact = (z0 + a * u / p) * torch.exp(p * t) - a * u / p     # analytic solution

print(f"final value   Euler = {z_euler[-1].item():.6f} | RK4 = {z_rk4[-1].item():.6f} | exact = {z_exact[-1].item():.6f}")
print(f"max |error|   Euler = {(z_euler - z_exact).abs().max().item():.3e} | RK4 = {(z_rk4 - z_exact).abs().max().item():.3e}")

# ---- trajectories + error -----------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax1.plot(t, z_exact, lw=2, color="#0b0b0b", label="analytic")
ax1.plot(t, z_rk4,   lw=1.5, ls="--", color="#2a78d6", label="RK4 (dt = 0.1)")
ax1.plot(t, z_euler, lw=1.5, ls=":", color="#eb6834", label="Euler (dt = 0.1)")
ax1.set_xlabel("t"); ax1.set_ylabel("z(t)")
ax1.set_title("dz/dt = p·z + a·u  —  trajectories")
ax1.legend(fontsize=9)

# skip t = 0, where both solvers are exact (log(0) is not plotted)
ax2.plot(t[1:], (z_euler[1:] - z_exact[1:]).abs(), lw=1.5, color="#eb6834", label="Euler error")
ax2.plot(t[1:], (z_rk4[1:]   - z_exact[1:]).abs(), lw=1.5, color="#2a78d6", label="RK4 error")
ax2.set_yscale("log")
ax2.set_xlabel("t"); ax2.set_ylabel("|error|")
ax2.set_title("error vs analytic solution (log scale)")
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   PART II · 2 — a GENERALISED system:  dz/dt = A·z + B·u(t)     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Same solve_ode machinery, but z is now an N-dim state and u(t) an
# M-dim time-varying input — N coupled differential equations:
#
#     dz1/dt = A11·z1 + A12·z2 + … + A1N·zN + B1·u(t)
#     dz2/dt = A21·z1 + A22·z2 + … + A2N·zN + B2·u(t)
#       ⋮
#     dzN/dt = AN1·z1 + … + ANN·zN + BN·u(t)
#
# Everything below is written generically as the single matrix equation
# dz/dt = A @ z + B @ u(t), so ANY N and M work — just change
# N_STATES / N_INPUTS. (make_stable_matrix, input_signal and rhs_system
# are reused by PART II · 4.) There is no closed form for a time-varying
# input, so both solvers are checked against an RK4 reference on a
# 10× finer grid.
import torch

N_STATES, N_INPUTS = 3, 2          # ⬅ the generalisation lives in these two numbers

def make_stable_matrix(n, margin=0.3):
    """Random n×n matrix with Re(λ) ≤ −margin (asymptotically stable)."""
    M = torch.randn(n, n)
    shift = torch.linalg.eigvals(M).real.max() + margin
    return M - shift * torch.eye(n)

def input_signal(t):
    """M-dim time-varying input: one sinusoid per channel, distinct freqs."""
    k = torch.arange(1, N_INPUTS + 1)
    return torch.sin(0.8 * k * t)

def rhs_system(t, z, A, B):        # dz/dt = A·z + B·u(t)
    return A @ z + B @ input_signal(t)

torch.manual_seed(0)               # reproducible random stable system
A = make_stable_matrix(N_STATES)
B = torch.randn(N_STATES, N_INPUTS) * 0.5
z0 = torch.randn(N_STATES) * 0.5

print(f"system: {N_STATES} state equations × {N_INPUTS} inputs,  "
      f"max Re(λ(A)) = {torch.linalg.eigvals(A).real.max().item():+.3f}  (stable if < 0)")

t     = torch.linspace(0, 12, 121)   # dt = 0.1
t_fin = torch.linspace(0, 12, 1201)  # 10× finer reference grid

z_euler = solve_ode(rhs_system, z0, t,     A, B, method="euler")
z_rk4   = solve_ode(rhs_system, z0, t,     A, B, method="rk4")
z_ref   = solve_ode(rhs_system, z0, t_fin, A, B, method="rk4")[::10]

print(f"max |error| vs fine RK4   Euler = {(z_euler - z_ref).abs().max().item():.3e}   RK4 = {(z_rk4 - z_ref).abs().max().item():.3e}")

# ---- state trajectories + phase portrait --------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.2))
colors = ["#2a78d6", "#eb6834", "#0b0b0b"]
for k in range(N_STATES):
    ax1.plot(t, z_rk4[:, k], lw=1.5, color=colors[k % len(colors)], label=f"z{k+1} — RK4")
ax1.plot(t, z_euler[:, 0], lw=1.5, ls=":", color="#c3c2b7", label="z1 — Euler")
ax1.set_xlabel("t"); ax1.set_ylabel("z(t)")
ax1.set_title(f"dz/dt = A·z + B·u(t) — {N_STATES} coupled state equations")
ax1.legend(fontsize=9)

# phase portrait: 2-D drift field when N = 2, a 3-D trajectory curve when N ≥ 3
if N_STATES == 2:
    z1g, z2g = torch.meshgrid(torch.linspace(-1.5, 1.5, 15),
                              torch.linspace(-1.5, 1.5, 15), indexing="ij")
    drift = A @ torch.stack([z1g.flatten(), z2g.flatten()])
    drift = drift / drift.norm(dim=0).clamp(min=1e-9)   # unit direction arrows
    ax2.quiver(z1g, z2g, drift[0].reshape(15, 15), drift[1].reshape(15, 15),
               color="#c3c2b7", angles="xy", scale_units="xy", scale=5, width=0.004)
    ax2.plot(z_euler[:, 0], z_euler[:, 1], lw=1.5, ls=":", color="#eb6834", label="Euler")
    ax2.plot(z_rk4[:, 0],   z_rk4[:, 1],   lw=1.5, ls="--", color="#2a78d6", label="RK4")
    ax2.plot(z0[0], z0[1], marker="o", ms=8, color="#0b0b0b", label="start")
    ax2.set_xlabel("z1"); ax2.set_ylabel("z2")
    ax2.set_title("phase portrait (arrows = drift A·z)")
else:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers "3d")
    ax2.remove()
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    ax2.plot(z_rk4[:, 0],   z_rk4[:, 1],   z_rk4[:, 2],   lw=1.5, color="#2a78d6", label="RK4")
    ax2.plot(z_euler[:, 0], z_euler[:, 1], z_euler[:, 2],
             lw=1.5, ls=":", color="#eb6834", label="Euler")
    ax2.plot([z0[0]], [z0[1]], [z0[2]], marker="o", ms=8, color="#0b0b0b", label="start")
    ax2.set_xlabel("z1"); ax2.set_ylabel("z2"); ax2.set_zlabel("z3")
    ax2.set_title(f"phase portrait ({N_STATES}-D trajectory)")
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   PART II · 3 — LEARN p and a from data: gradient descent        ║
# ║   THROUGH the RK4 solver (the ODE is nonlinear in p, a)         ║
# ╚══════════════════════════════════════════════════════════════════╝
# PART I's fitter needs equations that are linear in their parameters.
# dz/dt = p·z + a·u is NOT linear in p, a — but because solve_ode is a
# chain of differentiable torch ops, backprop flows through the whole
# simulated trajectory, so Adam can fit p, a like training a network.
import torch
import torch.nn as nn

torch.manual_seed(0)

# ---- 1. TARGET: the measured trajectory the ODE is fitted to --------------------
# (here: synthetic truth + noise; replace z_data with your own measurements)
p_true, a_true = -0.5, 1.0
t_obs = torch.linspace(0, 5, 26)
z_data = solve_ode(rhs_scalar, torch.tensor(1.0), t_obs, p_true, a_true, u) \
       + 0.01 * torch.randn(26)

# ---- 2. learn p, a with Adam --------------------------------------------------
params = nn.Parameter(torch.tensor([-0.1, 0.2]))    # deliberately bad guess
opt = torch.optim.Adam([params], lr=0.05)

for it in range(400):
    p, a = params[0], params[1]
    z_pred = solve_ode(rhs_scalar, torch.tensor(1.0), t_obs, p, a, u)
    loss = ((z_pred - z_data) ** 2).mean()
    opt.zero_grad()
    loss.backward()          # gradients flow back through 25 RK4 steps
    opt.step()
    if it % 100 == 0:
        print(f"  iter {it:3d}   p = {p.item():+.4f}   a = {a.item():+.4f}   loss = {loss.item():.3e}")

print(f"\n  true p = {p_true:+.1f}, a = {a_true:+.1f}"
      f"   |   recovered p = {params[0].item():+.4f}, a = {params[1].item():+.4f}")

# ---- 3. the fitted trajectory vs the measurements ------------------------------
z_fit = solve_ode(rhs_scalar, torch.tensor(1.0), t_obs, params[0], params[1], u)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(t_obs, z_data, s=26, color="#0b0b0b", edgecolors="#fcfcfb",
           linewidths=0.5, zorder=3, label="measurements (noisy)")
ax.plot(t_obs, z_fit.detach(), lw=2, color="#2a78d6", label="fitted solution (learned p, a)")
ax.set_xlabel("t"); ax.set_ylabel("z(t)")
ax.set_title("fitting dz/dt = p·z + a·u with gradient descent through RK4")
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [13]:
import matplotlib.pyplot as plt


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   PART II · 4 — GENERALISED mixed system: N ODEs + N linear eqs ║
# ╚══════════════════════════════════════════════════════════════════╝
# A full hidden state-space model. N HIDDEN states z(t) follow N
# coupled differential equations; N LINEAR algebraic equations map
# them to what we can measure (the gains c and the SHARED offset d
# are unknown, as in PART I):
#
#     dz/dt = A·z + B·u(t)          (N differential equations → solve_ode)
#     y1 = c1·z1 + d  …  yN = cN·zN + d    (N linear equations)
#
# The system is run TWICE from two known initial states (E = 2
# "experiments"). Why two? A single trajectory leaves the model
# identifiable only up to a similarity transform T with T·z0 = z0 —
# the scalar PART II · 3 had exactly one such T, pinned by z0 = 1.
# In N ≥ 2 dimensions the ambiguity is real, so two different z0's
# pin it down.
#
# Fitting is a hybrid of PART I and PART II · 3:
#   • c and d enter the output equations LINEARLY → exact least
#     squares (torch.linalg.lstsq) at every step, no gradient descent.
#   • A, B enter the ODE nonlinearly → Adam through the solver, as in
#     PART II · 3. (Variable projection: gradients flow through the
#     lstsq, whose optimality makes the implicit term vanish.)
# The landscape has local minima, so we restart a few times from
# different guesses and keep the run with the lowest loss.
import torch
import torch.nn as nn

torch.manual_seed(1)                        # reproducible truth + data

N = 3                                       # ⬅ dimension knobs: states...
E = 2                                       # ...and experiments
NOISE = 0.01
t_obs = torch.linspace(0, 10, 51)           # where the outputs are observed

# ---- 1. the true system -------------------------------------------------------
A_t = make_stable_matrix(N, margin=0.1)     # general N×N coupling (hidden)
B_t = torch.randn(N, N_INPUTS) * 0.5
c_t = torch.randn(N) * 0.8
d_t = 0.5 * torch.randn(1).item()
Z0 = torch.randn(E, N) * 0.5                # E KNOWN initial states
Z0[1] += 0.3                                #   (kept distinct)

def mixed_forward(t, A, B, c, d):
    zs, ys = [], []
    for z0e in Z0:
        z = solve_ode(rhs_system, z0e, t, A, B)     # N hidden states
        zs.append(z)
        ys.append(z * c + d)                        # N linear output equations
    return torch.stack(zs), torch.stack(ys)         # (E, T, N) each

t_sim = torch.linspace(0, 10, 201)
Z_true, Y_true = mixed_forward(t_sim, A_t, B_t, c_t, d_t)

# ---- 2. TARGET: the observed outputs, shape (E, len(t_obs), N) -----------------
# (here: synthetic truth + noise. For real data, replace Y_obs with your
#  measurements and set Z0 to the initial states the experiments actually
#  started from — then comment out the truth comparison in section 5.)
torch.manual_seed(101)
Y_obs = mixed_forward(t_obs, A_t, B_t, c_t, d_t)[1] + NOISE * torch.randn(E, 51, N)
print(f"noise floor (σ²) = {NOISE**2:.1e} — the target loss")

# ---- 3. the linear part: exact least squares for c, d (closed form) ------------
def fit_cd(Z, Y):
    """Least squares for y_i(t) = c_i·z_i(t) + d — differentiable, exact."""
    T, _ = Z.shape
    rows = []
    for i in range(N):
        R = torch.zeros(T, N + 1)
        R[:, i] = Z[:, i]
        R[:, -1] = 1.0
        rows.append(R)
    sol = torch.linalg.lstsq(torch.cat(rows), Y.T.reshape(-1)).solution
    return sol[:N], sol[-1]

# ---- 4. learn A, B with Adam + restarts (keep the best loss) -------------------
def loss_fn(par):
    A = par[:N*N].reshape(N, N)
    B = par[N*N:].reshape(N, N_INPUTS)
    Z = torch.cat([solve_ode(rhs_system, z0e, t_obs, A, B) for z0e in Z0])
    Y = Y_obs.reshape(-1, N)
    c, d = fit_cd(Z, Y)
    return ((Z * c + d - Y) ** 2).mean()

best_par, best_loss = None, float("inf")
torch.manual_seed(201)
for start in range(8):                     # local minima → keep the best run
    par = nn.Parameter(torch.cat([
        make_stable_matrix(N).flatten() * 0.3,
        torch.randn(N, N_INPUTS).flatten() * 0.3,
    ]))
    opt = torch.optim.Adam([par], lr=0.05)
    for it in range(500):
        loss = loss_fn(par)
        opt.zero_grad(); loss.backward(); opt.step()
    if loss.item() < best_loss:
        best_loss, best_par = loss.item(), par.detach().clone()
    print(f"  start {start}: final loss = {loss.item():.3e}"
          + ("  ← best" if loss.item() <= best_loss else ""))

A_f = best_par[:N*N].reshape(N, N)
B_f = best_par[N*N:].reshape(N, N_INPUTS)
Z_fit_obs = torch.cat([solve_ode(rhs_system, z0e, t_obs, A_f, B_f) for z0e in Z0])
c_f, d_f = fit_cd(Z_fit_obs, Y_obs.reshape(-1, N))

# ---- 5. report -------------------------------------------------------------------
print(f"\nbest loss = {best_loss:.2e}   (noise floor {NOISE**2:.1e})")
print(f"max |err|   A = {(A_f - A_t).abs().max().item():.3f}   "
      f"B = {(B_f - B_t).abs().max().item():.3f}   "
      f"c = {(c_f - c_t).abs().max().item():.3f}   "
      f"d = {abs(d_f.item() - d_t):.3f}")
torch.set_printoptions(precision=3, sci_mode=False)
print("\nA true:\n", A_t)
print("A recovered:\n", A_f)

# ---- 6. fitted outputs + the recovered HIDDEN states ----------------------------
Z_fit, Y_fit = mixed_forward(t_sim, A_f, B_f, c_f, d_f)

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8))
colors = ["#2a78d6", "#eb6834", "#0b0b0b"]
for e in range(E):
    ax = axes[0, e]
    for k in range(N):
        ax.scatter(t_obs, Y_obs[e, :, k], s=12, color=colors[k],
                   edgecolors="none", alpha=0.8)
        ax.plot(t_sim, Y_fit[e, :, k], lw=1.5, color=colors[k], label=f"y{k+1}")
    ax.set_xlabel("t"); ax.set_ylabel("y(t)")
    ax.set_title(f"experiment {e+1} — measured vs fitted outputs")
    ax.legend(fontsize=8)
for e in range(E):
    ax = axes[1, e]
    for k in range(N):
        ax.plot(t_sim, Z_true[e, :, k], lw=2, color=colors[k])
        ax.plot(t_sim, Z_fit[e, :, k], lw=1.5, ls="--", color=colors[k], label=f"z{k+1}")
    ax.set_xlabel("t"); ax.set_ylabel("z(t)")
    ax.set_title(f"experiment {e+1} — hidden state: true vs recovered")
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()